# تشخیص فرمان‌های اولتراسوند با Sherpa-ONNX

این نوت‌بوک جایگزین نسخه قبلی `other_commands.ipynb` است و فقط بخش تشخیص ۱۴ فرمان اولتراسوند را پوشش می‌دهد. نوت‌بوک `Hey_Nova.ipynb` خارج از محدوده این تغییر است و نباید تغییر کند.

## تفاوت بنیادین روش جدید

در روش قبلی، شبکه اختصاصی Adaptive PCEN + GRU + CNN با داده‌های پروژه آموزش داده می‌شد. در این نسخه از مدل ازپیش‌آموزش‌دیده انگلیسی Zipformer در Sherpa-ONNX استفاده می‌شود و عبارت‌های هدف با BPE به فایل Keywords تبدیل می‌شوند. بنابراین **این روش در این نوت‌بوک Fine-tuning یا آموزش مجدد وزن‌های مدل نیست**.

دیتاست فعلی از Google Drive خوانده و از نظر ساختار و سلامت فایل‌ها ممیزی می‌شود؛ داده‌ها برای ارزیابی اختصاصی و کالیبراسیون آستانه‌ها در مرحله بعد استفاده خواهند شد. پوشه `14_negative` یک Keyword نیست و برای ارزیابی آینده نرخ فعال‌سازی اشتباه حفظ می‌شود.

## ترتیب اجرا در Colab

سلول‌ها را از ابتدا تا انتها اجرا کنید. در اجرای اول، پکیج‌ها و مدل دانلود می‌شوند؛ مدل در Google Drive ذخیره می‌شود تا در نشست‌های بعدی دوباره دانلود نشود. در پایان، شیء `kws` آماده استفاده است. عمداً هیچ تست صوتی یا ارزیابی دقتی در این نسخه اجرا نمی‌شود؛ مرحله تست را جداگانه طراحی خواهیم کرد.


In [ ]:
%pip -q install sherpa-onnx sentencepiece pypinyin librosa soundfile

import importlib.metadata

for package_name in ["sherpa-onnx", "sentencepiece", "pypinyin", "librosa", "soundfile"]:
    try:
        print(f"{package_name}: {importlib.metadata.version(package_name)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package_name}: not found")


In [ ]:
from google.colab import drive
from pathlib import Path
import json
import os
import shutil
import subprocess
import urllib.request

import librosa
import numpy as np
import sherpa_onnx

drive.mount('/content/drive')

# Dataset location used by the previous project notebook
DATASET_PATH = Path('/content/drive/MyDrive/Ultrasound_Voice_Dataset')

# Keep the model separate from the dataset and persist it in Google Drive
MODEL_NAME = 'sherpa-onnx-kws-zipformer-gigaspeech-3.3M-2024-01-01'
MODEL_ROOT = Path('/content/drive/MyDrive/Sherpa_ONNX_Models')
MODEL_DIR = MODEL_ROOT / MODEL_NAME
MODEL_URL = (
    'https://github.com/k2-fsa/sherpa-onnx/releases/download/kws-models/'
    + MODEL_NAME + '.tar.bz2'
)

# Set True only if you explicitly want to download the pretrained model again.
FORCE_REDOWNLOAD_MODEL = False #@param {type:"boolean"}

# Preserve the exact class-folder names and order from the previous notebook.
CLASS_NAMES = [
    '00_capture_image',
    '01_capture_cine',
    '02_increase_depth',
    '03_decrease_depth',
    '04_increase_gain',
    '05_decrease_gain',
    '06_freeze',
    '07_toggle_auto_gain',
    '08_center_guide',
    '09_color_doppler',
    '10_power_doppler',
    '11_needle_enhance',
    '12_b_mode',
    '13_m_mode',
    '14_negative',
]

# Only the 14 positive command classes become KWS keywords.
# The spoken phrases are explicit so they are not guessed from folder names.
COMMANDS = [
    ('00_capture_image', 'CAPTURE IMAGE'),
    ('01_capture_cine', 'CAPTURE CINE'),
    ('02_increase_depth', 'INCREASE DEPTH'),
    ('03_decrease_depth', 'DECREASE DEPTH'),
    ('04_increase_gain', 'INCREASE GAIN'),
    ('05_decrease_gain', 'DECREASE GAIN'),
    ('06_freeze', 'FREEZE'),
    ('07_toggle_auto_gain', 'TOGGLE AUTO GAIN'),
    ('08_center_guide', 'CENTER GUIDE'),
    ('09_color_doppler', 'COLOR DOPPLER'),
    ('10_power_doppler', 'POWER DOPPLER'),
    ('11_needle_enhance', 'NEEDLE ENHANCE'),
    ('12_b_mode', 'B MODE'),
    ('13_m_mode', 'M MODE'),
]

NEGATIVE_CLASS = '14_negative'
AUDIO_EXTENSIONS = {'.wav', '.mp3', '.flac', '.m4a', '.ogg'}

# Sherpa model feature configuration (not the old 40-bin Mel input).
SAMPLE_RATE = 16000
FEATURE_DIM = 80

# Initial KWS decoding settings. These are starting values, not calibrated
# medical-safety thresholds; they must be evaluated in the later test phase.
NUM_THREADS = 2
PROVIDER = 'cpu'
MAX_ACTIVE_PATHS = 4
KEYWORDS_SCORE = 1.0
KEYWORDS_THRESHOLD = 0.25
NUM_TRAILING_BLANKS = 8

RAW_KEYWORDS_PATH = MODEL_DIR / 'ultrasound_commands_keywords_raw.txt'
KEYWORDS_PATH = MODEL_DIR / 'ultrasound_commands_keywords.txt'
MANIFEST_PATH = MODEL_DIR / 'ultrasound_kws_manifest.json'

assert len(COMMANDS) == 14, f'Expected 14 commands, got {len(COMMANDS)}'
assert len({class_name for class_name, _ in COMMANDS}) == len(COMMANDS)
assert NEGATIVE_CLASS in CLASS_NAMES

print(f'Dataset path: {DATASET_PATH}')
print(f'Model directory: {MODEL_DIR}')
print(f'Positive command keywords: {len(COMMANDS)}')
print(f'Negative folder (not a keyword): {NEGATIVE_CLASS}')


## ممیزی دیتاست فعلی

ساختار پوشه‌ها همان ساختار نوت‌بوک قبلی است. این سلول همه ۱۵ پوشه را بررسی می‌کند، فایل‌های صوتی قابل‌خواندن را می‌شمارد و فایل‌های خراب یا خالی را گزارش می‌دهد. این مرحله **آموزش مدل Sherpa نیست**؛ فقط اطمینان می‌دهد مسیر و داده‌های فعلی در Colab در دسترس‌اند.


In [ ]:
if not DATASET_PATH.is_dir():
    raise FileNotFoundError(
        f'Dataset folder not found: {DATASET_PATH}\n'
        'Check that Google Drive is mounted and the folder name is correct.'
    )

missing_folders = [
    class_name for class_name in CLASS_NAMES
    if not (DATASET_PATH / class_name).is_dir()
]
if missing_folders:
    raise FileNotFoundError(
        'Missing dataset class folders: ' + ', '.join(missing_folders)
    )

dataset_audit = []
problem_files = []

print(f'Dataset: {DATASET_PATH}\n')
for class_name in CLASS_NAMES:
    folder = DATASET_PATH / class_name
    files = sorted(
        path for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS
    )
    readable_count = 0
    durations = []

    for audio_path in files:
        try:
            waveform, source_rate = librosa.load(
                str(audio_path), sr=None, mono=True
            )
            if waveform.size == 0:
                raise ValueError('Empty audio file')
            if not np.isfinite(waveform).all():
                raise ValueError('Audio contains non-finite values')

            duration = float(waveform.size / source_rate)
            durations.append(duration)
            readable_count += 1
            dataset_audit.append({
                'class_name': class_name,
                'path': str(audio_path),
                'duration_seconds': duration,
                'sample_rate': int(source_rate),
            })
        except Exception as error:
            problem_files.append((str(audio_path), str(error)))

    print(
        f'{class_name:24s} | files: {len(files):4d} | '
        f'readable: {readable_count:4d}'
    )
    if not files:
        print('  WARNING: this class folder contains no supported audio files.')

print(f'\nTotal readable files: {len(dataset_audit)}')
print(f'Unreadable/empty files: {len(problem_files)}')

if dataset_audit:
    all_durations = [item['duration_seconds'] for item in dataset_audit]
    print(f'Shortest audio: {min(all_durations):.2f} s')
    print(f'Longest audio:  {max(all_durations):.2f} s')

if problem_files:
    print('\nFirst 20 files that could not be read:')
    for path, error in problem_files[:20]:
        print(f'{path} -> {error}')

if not dataset_audit:
    raise RuntimeError('No readable audio files were found in the dataset.')


## دریافت مدل ازپیش‌آموزش‌دیده

مدل انتخاب‌شده نسخه انگلیسی `Zipformer Transducer` آموزش‌دیده روی زیرمجموعه GigaSpeech XL است. فایل‌های مدل در Google Drive نگهداری می‌شوند. نسخه FP32 برای خط مبنای اولیه انتخاب شده است؛ اجرای مدل در این نوت‌بوک روی CPU تنظیم شده و به GPU وابسته نیست.


In [ ]:
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

REQUIRED_MODEL_FILES = [
    'encoder-epoch-12-avg-2-chunk-16-left-64.onnx',
    'decoder-epoch-12-avg-2-chunk-16-left-64.onnx',
    'joiner-epoch-12-avg-2-chunk-16-left-64.onnx',
    'tokens.txt',
    'bpe.model',
]

if FORCE_REDOWNLOAD_MODEL and MODEL_DIR.exists():
    shutil.rmtree(MODEL_DIR)

missing_model_files = [
    name for name in REQUIRED_MODEL_FILES
    if not (MODEL_DIR / name).is_file()
]

if missing_model_files:
    archive_path = Path('/content') / f'{MODEL_NAME}.tar.bz2'
    if archive_path.exists():
        archive_path.unlink()

    print('Downloading the official pretrained Sherpa-ONNX model...')
    urllib.request.urlretrieve(MODEL_URL, archive_path)

    extraction = subprocess.run(
        ['tar', '-xjf', str(archive_path), '-C', str(MODEL_ROOT)],
        capture_output=True,
        text=True,
    )
    if extraction.returncode != 0:
        raise RuntimeError('Model extraction failed:\n' + extraction.stderr)

    archive_path.unlink(missing_ok=True)

missing_model_files = [
    name for name in REQUIRED_MODEL_FILES
    if not (MODEL_DIR / name).is_file()
]
if missing_model_files:
    raise FileNotFoundError(
        'The model archive was extracted, but these files are missing: '
        + ', '.join(missing_model_files)
    )

print('Model files are ready:')
for name in REQUIRED_MODEL_FILES:
    file_path = MODEL_DIR / name
    print(f'  {name:58s} {file_path.stat().st_size / (1024 ** 2):7.2f} MB')


## تبدیل ۱۴ فرمان به فایل Keywords

مدل GigaSpeech از BPE استفاده می‌کند. بنابراین عبارت‌های انگلیسی باید با همان `bpe.model` و `tokens.txt` مدل به توکن تبدیل شوند. در انتهای هر خط، شناسه پوشه با پیشوند `@` قرار می‌گیرد تا نتیجه تشخیص به نام کلاس متناظر قابل نگاشت باشد.


In [ ]:
TOKENS_PATH = MODEL_DIR / 'tokens.txt'
BPE_MODEL_PATH = MODEL_DIR / 'bpe.model'

raw_keyword_lines = []
encoded_keyword_lines = []
KEYWORD_TO_CLASS = {}

for class_name, phrase in COMMANDS:
    encoded = sherpa_onnx.text2token(
        [phrase],
        tokens=str(TOKENS_PATH),
        tokens_type='bpe',
        bpe_model=str(BPE_MODEL_PATH),
    )

    if len(encoded) != 1 or not encoded[0]:
        raise ValueError(
            f'Could not tokenize command {phrase!r}. '
            'Check the model BPE files and command spelling.'
        )

    token_line = ' '.join(str(token) for token in encoded[0])
    raw_keyword_lines.append(f'{phrase} @{class_name}')
    encoded_keyword_lines.append(f'{token_line} @{class_name}')
    KEYWORD_TO_CLASS[class_name] = class_name

RAW_KEYWORDS_PATH.write_text(
    '\n'.join(raw_keyword_lines) + '\n', encoding='utf-8'
)
KEYWORDS_PATH.write_text(
    '\n'.join(encoded_keyword_lines) + '\n', encoding='utf-8'
)

if len(encoded_keyword_lines) != 14:
    raise RuntimeError(
        f'Expected 14 encoded keywords, got {len(encoded_keyword_lines)}'
    )

print(f'Created {len(encoded_keyword_lines)} keyword entries:')
print(KEYWORDS_PATH.read_text(encoding='utf-8'))


## ساخت موتور Keyword Spotting

در این مرحله فایل‌های ONNX، توکن‌های مدل و فایل Keywords به Sherpa-ONNX داده می‌شوند. `KEYWORDS_THRESHOLD` آستانه فعال‌شدن تشخیص است و نباید آن را معادل احتمال Softmax مدل CNN قبلی تفسیر کرد. مقدارهای زیر فقط تنظیمات اولیه‌اند و هنوز با داده‌های محیط بیمارستان کالیبره نشده‌اند.


In [ ]:
kws = sherpa_onnx.KeywordSpotter(
    tokens=str(TOKENS_PATH),
    encoder=str(MODEL_DIR / REQUIRED_MODEL_FILES[0]),
    decoder=str(MODEL_DIR / REQUIRED_MODEL_FILES[1]),
    joiner=str(MODEL_DIR / REQUIRED_MODEL_FILES[2]),
    keywords_file=str(KEYWORDS_PATH),
    num_threads=NUM_THREADS,
    sample_rate=SAMPLE_RATE,
    feature_dim=FEATURE_DIM,
    max_active_paths=MAX_ACTIVE_PATHS,
    keywords_score=KEYWORDS_SCORE,
    keywords_threshold=KEYWORDS_THRESHOLD,
    num_trailing_blanks=NUM_TRAILING_BLANKS,
    provider=PROVIDER,
)

print('Sherpa-ONNX KeywordSpotter initialized successfully.')
print(f'Model: {MODEL_NAME}')
print(f'Provider: {PROVIDER}')
print(f'Sample rate: {SAMPLE_RATE} Hz')
print(f'Feature dimension: {FEATURE_DIM}')
print(f'Keywords loaded: {len(COMMANDS)}')


## ذخیره مشخصات و پایان آماده‌سازی

فایل Manifest برای ثبت دقیق مدل، فرمان‌ها و پارامترهای استفاده‌شده ذخیره می‌شود. این فایل وزن آموزش‌دیده نیست؛ فقط مشخصات اجرای قابل‌بازتولید را نگه می‌دارد.


In [ ]:
manifest = {
    'framework': 'sherpa-onnx',
    'model_name': MODEL_NAME,
    'model_family': 'Zipformer Transducer Keyword Spotting',
    'model_language': 'English',
    'pretraining_corpus': 'GigaSpeech XL subset (approximately 10,000 hours)',
    'dataset_path': str(DATASET_PATH),
    'dataset_readable_files': len(dataset_audit),
    'dataset_unreadable_files': len(problem_files),
    'class_names': CLASS_NAMES,
    'negative_class': NEGATIVE_CLASS,
    'commands': [
        {'class_name': class_name, 'phrase': phrase}
        for class_name, phrase in COMMANDS
    ],
    'sample_rate': SAMPLE_RATE,
    'feature_dim': FEATURE_DIM,
    'provider': PROVIDER,
    'num_threads': NUM_THREADS,
    'max_active_paths': MAX_ACTIVE_PATHS,
    'keywords_score': KEYWORDS_SCORE,
    'keywords_threshold': KEYWORDS_THRESHOLD,
    'num_trailing_blanks': NUM_TRAILING_BLANKS,
    'training_performed': False,
    'fine_tuning_performed': False,
    'project_dataset_used_to_update_model_weights': False,
    'keywords_file': str(KEYWORDS_PATH),
}

MANIFEST_PATH.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print('\n' + '=' * 64)
print('READY: Sherpa-ONNX is configured for the 14 ultrasound commands.')
print('No model training or audio accuracy test was performed.')
print(f'Keywords file: {KEYWORDS_PATH}')
print(f'Manifest: {MANIFEST_PATH}')
print('=' * 64)


## مواردی که عمداً به مرحله بعد موکول شده‌اند

- اجرای مدل روی فایل‌های صوتی هر کلاس و محاسبه نرخ تشخیص صحیح.
- اجرای مدل روی پوشه `14_negative` و اندازه‌گیری False Activation.
- بررسی فرمان‌های آوایی نزدیک به هم مانند `CAPTURE IMAGE` و `CAPTURE CINE`.
- آزمایش میکروفون زنده و سنجش تأخیر واقعی.
- تنظیم `keywords_threshold`، `keywords_score` و `num_trailing_blanks` بر اساس داده و شرایط واقعی.

تا قبل از انجام این ارزیابی‌ها، خروجی این نوت‌بوک را نباید مدل تأییدشده برای کنترل دستگاه پزشکی تلقی کرد.
